# Liver Disease CNN

End-to-end Colab workflow for the **HCC / liver ultrasound CNN** in this repository.

This notebook covers everything related to the CNN pipeline:

| Repo component | What this notebook does |
|---|---|
| `dataset_CNN/` | Auto-clone from GitHub and prepare `dataset/` |
| `preprocessing/` | Resize, normalize, augment, visualize |
| `CNN_Model/` | Build custom CNN architecture |
| `models/` | Load pretrained or train & save models |
| `test_samples/` | Run inference on sample ultrasound images |

**Classes:** `normal`, `benign`, `malignant`  
**Input size:** `224 × 224 × 3`

---

## How to use in Colab

1. Upload this notebook to [Google Colab](https://colab.research.google.com/).
2. **Runtime → Change runtime type → GPU** (recommended).
3. Run cells top to bottom.
4. The notebook clones [azekowka/Liver-Disease-Diagnosis](https://github.com/azekowka/Liver-Disease-Diagnosis) and automatically prepares training images from `dataset_CNN/` (no manual upload needed).

5. To skip training, set `TRAIN_MODEL = False` and use the pretrained model already in the repo.

> **Tip:** Enable Google Drive in the setup cell to persist models between sessions.

## 1. Environment setup

In [ ]:
# Install dependencies (Colab usually has TF, but we ensure versions are present)
!pip install -q tensorflow opencv-python-headless pillow scikit-learn matplotlib seaborn pandas

In [ ]:
import os
import sys
import pickle
import shutil
import zipfile
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {len(gpus)} device(s)')
    for gpu in gpus:
        print(' ', gpu)
else:
    print('No GPU detected — training will run on CPU (slow). Go to Runtime → Change runtime type → GPU.')

## 2. Configuration

Adjust these flags before running the rest of the notebook.

In [ ]:
# ── Runtime mode ──────────────────────────────────────────────────────────────
USE_GOOGLE_DRIVE = False          # Set True to mount Drive and persist files
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Liver-Disease-Diagnosis'

CLONE_FROM_GITHUB = True          # Clone repo (dataset_CNN, models, test_samples)
GITHUB_REPO_URL = 'https://github.com/azekowka/Liver-Disease-Diagnosis.git'

# ── Dataset ───────────────────────────────────────────────────────────────────
PREPARE_DATASET_FROM_REPO = True  # Build dataset/ from dataset_CNN/ after clone
SKIP_PREPROCESSING = False        # Set True if dataset_augmented/ already exists

# ── Training ──────────────────────────────────────────────────────────────────
TRAIN_MODEL = True                # Set False to only run inference with pretrained weights
RESUME_TRAINING = True            # Load models/custom_liver_cnn if it exists
EPOCHS = 100
BATCH_SIZE = 32                   # Lower to 16 or 8 if GPU runs out of memory
VALIDATION_SPLIT = 0.30           # 70% train / 30% validation
AUGMENTATION_MULTIPLIER = 3       # Augmented images per source image

# ── Paths (relative to project root) ──────────────────────────────────────────
PROJECT_ROOT = Path('/content/Liver-Disease-Diagnosis')
DATASET_CNN = PROJECT_ROOT / 'dataset_CNN'
DATASET_RAW = PROJECT_ROOT / 'dataset'
DATASET_AUGMENTED = PROJECT_ROOT / 'dataset_augmented'
MODEL_CUSTOM = PROJECT_ROOT / 'models' / 'custom_liver_cnn'
MODEL_ALT = PROJECT_ROOT / 'models' / 'liver_classification'
TEST_SAMPLES = PROJECT_ROOT / 'test_samples'
HISTORY_PATH = PROJECT_ROOT / 'models' / 'training_history.pkl'

CATEGORIES = ['normal', 'benign', 'malignant']
IMG_SIZE = (224, 224)

print('Configuration loaded.')
print('Project root:', PROJECT_ROOT)

In [ ]:
# Optional: mount Google Drive for persistence
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    if Path(DRIVE_PROJECT_DIR).exists():
        PROJECT_ROOT = Path(DRIVE_PROJECT_DIR)
        print('Using existing Drive project:', PROJECT_ROOT)
    else:
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        print('Created Drive project folder:', PROJECT_ROOT)

    # Re-bind paths after Drive switch
    DATASET_CNN = PROJECT_ROOT / 'dataset_CNN'
    DATASET_RAW = PROJECT_ROOT / 'dataset'
    DATASET_AUGMENTED = PROJECT_ROOT / 'dataset_augmented'
    MODEL_CUSTOM = PROJECT_ROOT / 'models' / 'custom_liver_cnn'
    MODEL_ALT = PROJECT_ROOT / 'models' / 'liver_classification'
    TEST_SAMPLES = PROJECT_ROOT / 'test_samples'
    HISTORY_PATH = PROJECT_ROOT / 'models' / 'training_history.pkl'
else:
    print('Google Drive not mounted — files live in /content (lost when runtime ends).')

In [ ]:
# Clone repository (gets dataset_CNN, models, test_samples, CNN code)
if CLONE_FROM_GITHUB:
    if not PROJECT_ROOT.exists():
        !git clone {GITHUB_REPO_URL} {PROJECT_ROOT}
    else:
        print('Repo already exists at', PROJECT_ROOT)
        !cd {PROJECT_ROOT} && git pull
else:
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Set CLONE_FROM_GITHUB=True to fetch the repo automatically.')

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Top-level contents:', sorted(os.listdir(PROJECT_ROOT)))
print('dataset_CNN present:', (PROJECT_ROOT / 'dataset_CNN').exists())

## 3. Prepare training dataset

Automatically loads `dataset_CNN/` from the cloned GitHub repo and prepares:

```text
dataset/
  normal/     <- dataset_CNN/Normal/Normal/image/
  benign/     <- dataset_CNN/Benign/Benign/image/
  malignant/  <- dataset_CNN/Malignant/Malignant/image/
```

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
DATASET_CNN_CLASS_MAP = {
    'Normal': 'normal',
    'Benign': 'benign',
    'Malignant': 'malignant',
}


def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def prepare_dataset_from_repo(force=False):
    """Copy ultrasound images from dataset_CNN/ into dataset/{class}/."""
    if not DATASET_CNN.exists():
        raise FileNotFoundError(
            f'dataset_CNN not found at {DATASET_CNN}. '
            'Make sure CLONE_FROM_GITHUB=True and the repo contains dataset_CNN/.'
        )

    if count_images(DATASET_RAW) > 0 and not force:
        print('dataset/ already prepared — skipping copy (set force=True to rebuild).')
        return

    print('Preparing dataset/ from dataset_CNN/ ...')
    for src_class, dst_class in DATASET_CNN_CLASS_MAP.items():
        src_dir = DATASET_CNN / src_class / src_class / 'image'
        dst_dir = DATASET_RAW / dst_class
        dst_dir.mkdir(parents=True, exist_ok=True)

        if not src_dir.exists():
            print(f'  WARNING: missing source folder {src_dir}')
            continue

        for old_file in dst_dir.iterdir():
            if old_file.is_file():
                old_file.unlink()

        copied = 0
        for img_path in sorted(src_dir.iterdir()):
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            shutil.copy2(img_path, dst_dir / img_path.name)
            copied += 1
        print(f'  {dst_class:10s}: {copied:4d} images <- {src_dir}')

    print('Dataset preparation complete.')


if PREPARE_DATASET_FROM_REPO:
    prepare_dataset_from_repo()
else:
    for cat in CATEGORIES:
        (DATASET_RAW / cat).mkdir(parents=True, exist_ok=True)

print('\nDataset summary:')
for cat in CATEGORIES:
    n = count_images(DATASET_RAW / cat)
    print(f'  {cat:10s}: {n:4d} images')
print('  TOTAL     :', count_images(DATASET_RAW), 'images')

if count_images(DATASET_RAW) == 0:
    print('\n⚠️  No training images found. Check that dataset_CNN/ exists in the cloned repo.')
elif count_images(DATASET_RAW) < 50:
    print('\n⚠️  Very few images detected — verify dataset_CNN was cloned correctly.')

## 4. Image preprocessing (`preprocessing/`)

Mirrors the repo scripts:
- `resize_images.py`
- `normalize_images.py`
- `data_augmentation.py`
- `check_augmanted.py`

Output is written to `dataset_augmented/` (unified path used by training).

In [ ]:
def resize_dataset(dataset_path=DATASET_RAW, output_size=IMG_SIZE):
    """Equivalent to preprocessing/resize_images.py"""
    print('Resizing images to', output_size)
    for category in CATEGORIES:
        path = dataset_path / category
        if not path.exists():
            print(f'  Skipping missing folder: {path}')
            continue
        for img_file in path.iterdir():
            if img_file.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}:
                continue
            image = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
            if image is None:
                print(f'  Could not read {img_file}')
                continue
            image = cv2.resize(image, output_size)
            cv2.imwrite(str(img_file), image)
    print('Resize complete.')


def normalize_dataset(dataset_path=DATASET_RAW):
    """Equivalent to preprocessing/normalize_images.py"""
    print('Normalizing images to [0, 1] then saving as uint8')
    for category in CATEGORIES:
        path = dataset_path / category
        if not path.exists():
            continue
        for img_file in path.iterdir():
            if img_file.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}:
                continue
            image = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            image = image / 255.0
            cv2.imwrite(str(img_file), (image * 255).astype(np.uint8))
    print('Normalization complete.')


def augment_dataset(
    input_path=DATASET_RAW,
    output_path=DATASET_AUGMENTED,
    multiplier=AUGMENTATION_MULTIPLIER,
):
    """Equivalent to preprocessing/data_augmentation.py (outputs to dataset_augmented/)"""
    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    datagen = ImageDataGenerator(
        rotation_range=40,
        width_shift_range=0.3,
        height_shift_range=0.3,
        shear_range=0.3,
        zoom_range=0.3,
        brightness_range=[0.6, 1.5],
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='nearest',
    )

    total_saved = 0
    for category in CATEGORIES:
        input_folder = input_path / category
        output_folder = output_path / category
        output_folder.mkdir(parents=True, exist_ok=True)

        if not input_folder.exists():
            print(f'  Skipping missing class folder: {input_folder}')
            continue

        for img_file in input_folder.iterdir():
            if img_file.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}:
                continue
            try:
                image = Image.open(img_file).convert('RGB').resize(IMG_SIZE)
                image = np.array(image).astype(np.float32) / 255.0
                image = np.expand_dims(image, axis=0)
                aug_iter = datagen.flow(image, batch_size=1)

                # Keep original copy in augmented set
                Image.open(img_file).convert('RGB').resize(IMG_SIZE).save(output_folder / img_file.name)
                total_saved += 1

                for i in range(multiplier):
                    aug_image = next(aug_iter)[0]
                    aug_image = np.clip(aug_image * 255, 0, 255).astype(np.uint8)
                    out_name = output_folder / f'aug_{i}_{img_file.stem}.png'
                    Image.fromarray(aug_image).save(out_name, format='PNG')
                    total_saved += 1
            except Exception as exc:
                print(f'  ERROR processing {img_file}: {exc}')

    print(f'Augmentation complete. Saved {total_saved} images to {output_path}')


def show_augmented_samples(class_name='benign', n=6):
    """Equivalent to preprocessing/check_augmanted.py"""
    folder = DATASET_AUGMENTED / class_name
    if not folder.exists() or count_images(folder) == 0:
        print(f'No augmented images in {folder}')
        return

    files = [p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}][:n]
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for ax, img_path in zip(axes, files):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name[:20])
        ax.axis('off')
    plt.suptitle(f'Augmented samples — {class_name}', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
if count_images(DATASET_RAW) > 0:
    if SKIP_PREPROCESSING and DATASET_AUGMENTED.exists() and count_images(DATASET_AUGMENTED) > 0:
        print('Skipping preprocessing — using existing dataset_augmented/')
    else:
        resize_dataset()
        normalize_dataset()
        augment_dataset()
        show_augmented_samples('benign')
        show_augmented_samples('malignant')
        show_augmented_samples('normal')
else:
    print('Skipping preprocessing — no raw dataset available.')

## 5. Build CNN model (`CNN_Model/model.py`)

Custom CNN architecture:
- 4 convolutional blocks (32 → 64 → 128 → 256 filters)
- BatchNorm + LeakyReLU + MaxPooling
- GlobalAveragePooling2D
- Dense(256) + Dropout(0.5)
- Softmax output (3 classes)

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=3):
    """Mirrors CNN_Model/model.py"""
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.1),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.1),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.1),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.1),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(256),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.1),
        layers.Dropout(0.5),

        layers.Dense(num_classes, activation='softmax'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


preview_model = build_custom_cnn()
preview_model.summary()

## 6. Create data generators (`preprocessing/load_dataset.py`)

In [ ]:
train_generator = None
val_generator = None
class_indices = None

if count_images(DATASET_AUGMENTED) > 0:
    datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        validation_split=VALIDATION_SPLIT,
    )

    train_generator = datagen.flow_from_directory(
        str(DATASET_AUGMENTED),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        shuffle=True,
    )

    val_generator = datagen.flow_from_directory(
        str(DATASET_AUGMENTED),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        shuffle=False,
    )

    class_indices = train_generator.class_indices
    idx_to_class = {v: k for k, v in class_indices.items()}

    print('Class label mapping:', class_indices)
    print('Training samples:', train_generator.samples)
    print('Validation samples:', val_generator.samples)
else:
    print('No augmented dataset — skipping generator setup.')

## 7. Train or load pretrained model (`CNN_Model/train.py` + `models/`)

- `models/custom_liver_cnn/` — primary model from this repo
- `models/liver_classification/` — alternate saved model

Set `TRAIN_MODEL = False` to skip training and load pretrained weights only.

In [ ]:
model = None
history = None


def load_saved_model():
    if MODEL_CUSTOM.exists():
        print('Loading pretrained model:', MODEL_CUSTOM)
        return tf.keras.models.load_model(str(MODEL_CUSTOM))
    if MODEL_ALT.exists():
        print('Loading alternate model:', MODEL_ALT)
        return tf.keras.models.load_model(str(MODEL_ALT))
    return None


if TRAIN_MODEL and train_generator is not None:
    if RESUME_TRAINING and MODEL_CUSTOM.exists():
        print('Resuming training from saved checkpoint...')
        model = tf.keras.models.load_model(str(MODEL_CUSTOM))
    else:
        print('Training from scratch...')
        model = build_custom_cnn()

    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ]

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    MODEL_CUSTOM.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(MODEL_CUSTOM))
    print('Model saved to', MODEL_CUSTOM)

    with open(HISTORY_PATH, 'wb') as f:
        pickle.dump(history.history, f)
    print('Training history saved to', HISTORY_PATH)

else:
    model = load_saved_model()
    if model is None:
        print('No pretrained model found and training is disabled.')
    else:
        print('Loaded model for inference.')
        model.summary()

## 8. Visualize training history

In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
elif HISTORY_PATH.exists():
    with open(HISTORY_PATH, 'rb') as f:
        saved_history = pickle.load(f)
    print('Loaded saved history keys:', saved_history.keys())
else:
    print('No training history available.')

## 9. Evaluate on validation set

In [ ]:
if model is not None and val_generator is not None:
    val_generator.reset()
    y_prob = model.predict(val_generator, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = val_generator.classes

    target_names = [idx_to_class[i] for i in sorted(idx_to_class)]

    print('\nClassification report:')
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix — Validation Set')
    plt.tight_layout()
    plt.show()
else:
    print('Skipping evaluation — model or validation data unavailable.')

## 10. Inference on `test_samples/`

Run predictions on the sample ultrasound images included in the repository.

In [ ]:
def preprocess_image_for_prediction(img_path, target_size=IMG_SIZE):
    img = load_img(img_path, target_size=target_size)
    arr = img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)


def predict_image(model, img_path, class_indices_map=None):
    x = preprocess_image_for_prediction(img_path)
    probs = model.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(probs))

    if class_indices_map:
        idx_to_name = {v: k for k, v in class_indices_map.items()}
        pred_label = idx_to_name[pred_idx]
    else:
        pred_label = f'class_{pred_idx}'

    return pred_label, probs, pred_idx


def show_prediction(model, img_path, class_indices_map=None):
    label, probs, idx = predict_image(model, img_path, class_indices_map)

    img = Image.open(img_path).convert('RGB')
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{Path(img_path).name}\nPredicted: {label} ({probs[idx]:.2%})')
    plt.show()

    if class_indices_map:
        names = [k for k, _ in sorted(class_indices_map.items(), key=lambda x: x[1])]
        for name, p in zip(names, probs):
            print(f'  {name:10s}: {p:.4f}')
    else:
        print('Probabilities:', probs)

    return label, probs

In [ ]:
if model is not None and TEST_SAMPLES.exists():
    sample_files = sorted([
        p for p in TEST_SAMPLES.iterdir()
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    ])

    print(f'Found {len(sample_files)} test sample images\n')

    # Use class_indices from training if available; otherwise use alphabetical fallback
    indices = class_indices if class_indices else {'benign': 0, 'malignant': 1, 'normal': 2}

    results = []
    for img_path in sample_files:
        label, probs, idx = predict_image(model, img_path, indices)
        results.append({'file': img_path.name, 'prediction': label, 'confidence': float(probs[idx])})
        print(f'{img_path.name:45s} -> {label:10s} ({probs[idx]:.2%})')

    results_df = __import__('pandas').DataFrame(results)
    display(results_df)
else:
    print('Model or test_samples/ not available.')

In [ ]:
# Visualize a few test sample predictions
if model is not None and TEST_SAMPLES.exists():
    indices = class_indices if class_indices else {'benign': 0, 'malignant': 1, 'normal': 2}
    demo_files = sorted(TEST_SAMPLES.glob('*'))[:6]
    demo_files = [p for p in demo_files if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]

    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    for ax, img_path in zip(axes.flatten(), demo_files):
        label, probs, idx = predict_image(model, img_path, indices)
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'{label}\n{probs[idx]:.1%}', fontsize=10)
        ax.axis('off')
    plt.suptitle('Test sample predictions', fontsize=14)
    plt.tight_layout()
    plt.show()

## 11. Predict on your own image

Upload any liver ultrasound image for a single prediction.

In [ ]:
if model is not None:
    from google.colab import files

    print('Upload a liver ultrasound image (jpg/png)')
    uploaded = files.upload()
    custom_path = Path('/content') / next(iter(uploaded))
    custom_path.write_bytes(uploaded[custom_path.name])

    indices = class_indices if class_indices else {'benign': 0, 'malignant': 1, 'normal': 2}
    show_prediction(model, custom_path, indices)
else:
    print('Load or train a model first.')

## 12. Download trained artifacts

Zip and download the trained model and training history to your computer or Google Drive.

In [ ]:
def zip_directory(source_dir, zip_path):
    source_dir = Path(source_dir)
    if not source_dir.exists():
        print(f'Skipping missing folder: {source_dir}')
        return False
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in source_dir.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(source_dir.parent))
    print('Created', zip_path)
    return True


artifacts_zip = PROJECT_ROOT / 'cnn_artifacts.zip'
with zipfile.ZipFile(artifacts_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    if MODEL_CUSTOM.exists():
        for fp in MODEL_CUSTOM.rglob('*'):
            if fp.is_file():
                zf.write(fp, fp.relative_to(PROJECT_ROOT))
    if HISTORY_PATH.exists():
        zf.write(HISTORY_PATH, HISTORY_PATH.relative_to(PROJECT_ROOT))

print('Artifacts zip ready:', artifacts_zip)

from google.colab import files
files.download(str(artifacts_zip))

---

## Quick reference

| Goal | What to set |
|---|---|
| Only test pretrained model | `TRAIN_MODEL = False` |
| Train from scratch | `TRAIN_MODEL = True` (dataset auto-loaded from repo) |
| Rebuild dataset/ from dataset_CNN | `prepare_dataset_from_repo(force=True)` |
| Resume training | `TRAIN_MODEL = True`, `RESUME_TRAINING = True` |
| Skip preprocessing | `SKIP_PREPROCESSING = True` (needs existing `dataset_augmented/`) |
| Save between sessions | `USE_GOOGLE_DRIVE = True` |
| GPU OOM error | Lower `BATCH_SIZE` to `16` or `8` |

**Repo mapping:**
- `dataset_CNN/` → Section 3 `prepare_dataset_from_repo()`
- `preprocessing/resize_images.py` → Section 4 `resize_dataset()`
- `preprocessing/normalize_images.py` → Section 4 `normalize_dataset()`
- `preprocessing/data_augmentation.py` → Section 4 `augment_dataset()`
- `preprocessing/load_dataset.py` → Section 6 generators
- `CNN_Model/model.py` → Section 5 `build_custom_cnn()`
- `CNN_Model/train.py` → Section 7 training cell
- `models/custom_liver_cnn/` → saved weights
- `test_samples/` → Section 10 inference